In [4]:
# 1. Install dependencies
%pip -q install gradio seaborn

# 2. Imports and Configuration
import json
import os
import zipfile
from pathlib import Path

import gradio as gr
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tensorflow as tf

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 8
DATASET_DIR = "/content/cattle_breeds"
DATASET_ZIP = "/content/drive/MyDrive/Mini_Project/archive (1).zip"  # Targeting your Drive path
MODEL_DIR = "/content/cattle_breed_artifacts"

tf.keras.utils.set_random_seed(SEED)
AUTOTUNE = tf.data.AUTOTUNE

print("TensorFlow:", tf.__version__)
print("Gradio:", gr.__version__)

# 3. Mount Drive and Extract Dataset
MOUNT_DRIVE = True

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

def prepare_dataset_root(dataset_dir: str, dataset_zip: str = "") -> str:
    dataset_path = Path(dataset_dir)

    if dataset_zip:
        zip_path = Path(dataset_zip)
        if not zip_path.exists():
            raise FileNotFoundError(f"Zip not found: {zip_path}. Double check your file path!")

        dataset_path.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path, "r") as archive:
            archive.extractall(dataset_path)

        subdirs = [path for path in dataset_path.iterdir() if path.is_dir()]
        if len(subdirs) == 1:
            possible_root = subdirs[0]
            breed_dirs = [path for path in possible_root.iterdir() if path.is_dir()]
            if len(breed_dirs) >= 2:
                dataset_path = possible_root

    if not dataset_path.exists():
        raise FileNotFoundError(
            f"Dataset folder not found: {dataset_path}. Set DATASET_DIR or DATASET_ZIP first."
        )

    breed_dirs = [path for path in dataset_path.iterdir() if path.is_dir()]
    if len(breed_dirs) < 2:
        raise ValueError("Need at least two breed folders inside the dataset directory.")

    return str(dataset_path)

DATASET_DIR = prepare_dataset_root(DATASET_DIR, DATASET_ZIP)
print("Dataset ready at:", DATASET_DIR)

# 4. Load Dataset
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
num_classes = len(class_names)

train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

# 5. Build and Train Model
data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.05),
        tf.keras.layers.RandomZoom(0.12),
        tf.keras.layers.RandomContrast(0.1),
    ],
    name="augmentation"
)

base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMAGE_SIZE + (3,),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False

inputs = tf.keras.Input(shape=IMAGE_SIZE + (3,))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.25)(x)
outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs, name="cattle_breed_classifier")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=2),
]

print("Starting training...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

# 6. Save Model
os.makedirs(MODEL_DIR, exist_ok=True)
model_path = os.path.join(MODEL_DIR, "cattle_breed_model.keras")
labels_path = os.path.join(MODEL_DIR, "breed_labels.json")

model.save(model_path)
with open(labels_path, "w", encoding="utf-8") as file:
    json.dump(class_names, file, indent=2)

print("Saved model to:", model_path)

# 7. Launch UI
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def build_example_list(dataset_dir: str, limit: int = 4):
    example_paths = []
    for breed_dir in sorted(path for path in Path(dataset_dir).iterdir() if path.is_dir()):
        for image_path in breed_dir.rglob("*"):
            if image_path.suffix.lower() in IMAGE_EXTENSIONS:
                example_paths.append([str(image_path)])
                break
        if len(example_paths) >= limit:
            break
    return example_paths

def predict_breed(image):
    if image is None:
        return {"No image": 1.0}, "Upload a cattle photo to get a prediction."

    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.cast(image, tf.float32)
    predictions = model.predict(tf.expand_dims(image, axis=0), verbose=0)[0]
    top_indices = np.argsort(predictions)[::-1][:3]
    result = {class_names[index]: float(predictions[index]) for index in top_indices}
    best_label = class_names[top_indices[0]]
    best_score = float(predictions[top_indices[0]])
    message = f"Top match: {best_label} ({best_score:.1%} confidence)"
    return result, message

examples = build_example_list(DATASET_DIR, limit=4)

with gr.Blocks(theme=gr.themes.Base()) as demo:
    gr.Markdown("# Cattle Breed Console\nUpload a cattle image and the model will estimate the most likely breed.")

    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(type="numpy", label="Cattle photo")
            classify_button = gr.Button("Classify breed")

        with gr.Column(scale=1):
            prediction_output = gr.Label(num_top_classes=3, label="Breed odds")
            console_output = gr.Textbox(label="Console", interactive=False)

    if examples:
        gr.Markdown("### Sample cattle photos")
        gr.Examples(examples=examples, inputs=image_input)

    classify_button.click(
        fn=predict_breed,
        inputs=image_input,
        outputs=[prediction_output, console_output]
    )

demo.launch(debug=False, share=True)

TensorFlow: 2.19.0
Gradio: 5.50.0
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset ready at: /content/cattle_breeds/dataset
Found 3750 files belonging to 15 classes.
Using 3000 files for training.
Found 3750 files belonging to 15 classes.
Using 750 files for validation.
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Starting training...
Epoch 1/8
94/94 ━━━━━━━━━━━━━━━━━━━━ 236s 2s/step - accuracy: 0.3303 - loss: 2.1455 - val_accuracy: 0.5133 - val_loss: 1.5984 - learning_rate: 0.0010
Epoch 2/8
94/94 ━━━━━━━━━━━━━━━━━━━━ 228s 2s/step - accuracy: 0.5377 - loss: 1.4542 - val_accuracy: 0.5787 - val_loss: 1.3515 - learning_rate: 0.0010
Epoch 3/8
94/94 ━━━━━━━━━━━━━━━━━━━━ 202s 2s/step - accuracy: 0.5923 - loss: 1.2754 - val_accuracy: 0.6253 - val_loss: 1.2371 - learning_rate: 0.0010
Epoch 4/8
94/94 ━━━━━━━━━━━━━━━━━━━━ 205s 2s/step - accuracy: 0.6437 - loss: 1.1372 - val_accuracy: 0.6400 - val_loss: 1.

/tmp/ipykernel_17667/2281717135.py:182: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Base()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2224e085561d1e31c2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
